In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
# Path
PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from utils.data_loading import load_data as load_v1_original, load_test_data
from utils.data_loadingV2 import load_data as load_v2_temporal
from utils.data_loadingV3_random import load_data as load_v3_random
from utils.data_loadingV4_file import load_data as load_v4_file
from utils.simple_lds_loss import SimpleLDSLoss
from utils.fds import FDS

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

In [3]:
CONFIG = {
    'TRAIN_PATH': PROJECT_ROOT / "datasets_challenge" / "train",
    'VAL_RATIO': 0.2,
    'BATCH_SIZE': 64,
    'COLUMNS_TO_DROP': ['sideSlip', 'vyCG', 'longitude', 'latitude', 'time'],
    'SEQUENCE_LENGTH': 20,
    'THRESHOLD': True,
    'THRESHOLD_VALUE': 3.0,
    'THRESHOLD_COLUMN': 'vxCG',
    'TARGET': 'sideSlip',
    'SCALING': True,
    'SEED': 42,
    'NUM_BINS': 100,
    'DIST_CSV': PROJECT_ROOT / "datasets_challenge" / "sideSlip_distribution.csv",
}

In [4]:
print("Loading V1: Original Split")
train_v1, val_v1, pipeline_v1 = load_v1_original(
    train_path=CONFIG['TRAIN_PATH'],
    train_ratio=0.8,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold=CONFIG['THRESHOLD'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

print("\nLoading V2: Temporal Split")
train_v2, val_v2, scaler_v2, _, _ = load_v2_temporal(
    train_path=CONFIG['TRAIN_PATH'],
    val_ratio=0.15,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

print("\nLoading V3: Random Split")
train_v3, val_v3, scaler_v3, _, _ = load_v3_random(
    train_path=CONFIG['TRAIN_PATH'],
    val_ratio=0.2,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

print("\nLoading V4: File-Based Split (File 7)")
train_v4, val_v4, scaler_v4, _, _ = load_v4_file(
    train_path=CONFIG['TRAIN_PATH'],
    val_file_num=17,
    columns_to_drop=CONFIG['COLUMNS_TO_DROP'],
    sequence_length=CONFIG['SEQUENCE_LENGTH'],
    threshold_value=CONFIG['THRESHOLD_VALUE'],
    threshold_column=CONFIG['THRESHOLD_COLUMN'],
    target=CONFIG['TARGET'],
    batchsize=CONFIG['BATCH_SIZE'],
    seed=CONFIG['SEED']
)

def extract_targets(loader):
    targets = []
    for _, y in loader:
        targets.append(y)
    return torch.cat(targets).numpy().flatten()

train_targets_v1 = extract_targets(train_v1)
val_targets_v1 = extract_targets(val_v1)

train_targets_v2 = extract_targets(train_v2)
val_targets_v2 = extract_targets(val_v2)

train_targets_v3 = extract_targets(train_v3)
val_targets_v3 = extract_targets(val_v3)

train_targets_v4 = extract_targets(train_v4)
val_targets_v4 = extract_targets(val_v4)

print(f"\nDataset Sizes:")
print(f"V1 Original: Train {len(train_targets_v1):,} | Val {len(val_targets_v1):,}")
print(f"V2 Temporal: Train {len(train_targets_v2):,} | Val {len(val_targets_v2):,}")
print(f"V3 Random:   Train {len(train_targets_v3):,} | Val {len(val_targets_v3):,}")
print(f"V4 File:     Train {len(train_targets_v4):,} | Val {len(val_targets_v4):,}")

Loading V1: Original Split
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_001.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_002.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_003.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_004.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_005.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_006.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_007.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_008.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_009.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_010.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_011.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_012.csv
c:\mait_g2_drift_challenge\datasets_challenge\train\train_part_013.csv
c:\mait_g2_drift_challenge\datasets_challenge\trai

In [5]:
class VanillaRNN(nn.Module):
    """
    Vanilla RNN backbone that exposes penultimate features
    for FDS calibration.
    """
    def __init__(self, input_size=22, hidden_size=64, num_layers=1, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_size

        self.rnn = nn.RNN(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            nonlinearity='relu',
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # Regression head
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, return_features=False):
        _, h = self.rnn(x)           # h: (num_layers, B, hidden)
        feats = self.dropout(h[-1]) # (B, hidden_size)

        if return_features:
            return feats

        return self.regressor(feats)


In [6]:
def make_bin_edges(targets, num_bins, device):
    """
    Create uniform bin edges from TRAIN targets only.
    """
    min_y = float(targets.min())
    max_y = float(targets.max())
    return torch.linspace(min_y, max_y, num_bins + 1).to(device)


In [7]:
def train_eval_with_fds(
    train_loader,
    val_loader,
    train_targets,
    config,
    epochs=30
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Model
    model = VanillaRNN(
        input_size=22,
        hidden_size=48,
        num_layers=1,
        dropout=0.0
    ).to(device)

    # FDS
    fds = FDS(
        feat_dim=48,
        num_bins=config['NUM_BINS'],
        kernel='gaussian',
        ks=5,
        sigma=2.0,
        momentum=0.9,
        start_update=1,
        start_smooth=2
    ).to(device)

    # Bin edges (TRAIN ONLY)
    bin_edges = make_bin_edges(
        train_targets,
        config['NUM_BINS'],
        device
    )

    criterion = nn.L1Loss()  # MAE
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

    # =======================
    #  Training loop
    # =======================
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device).float()

            optimizer.zero_grad()

            # Penultimate features
            feats = model(x, return_features=True)

            # FDS calibration
            feats_cal = fds(
                feats=feats,
                y=y,
                bin_edges=bin_edges,
                epoch=epoch,
                train=True
            )

            # Regression
            preds = model.regressor(feats_cal).squeeze(-1)
            loss = criterion(preds, y)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        # Smooth stats once per epoch
        fds.update_last_epoch_stats(epoch)

        print(
            f"Epoch {epoch:02d} | "
            f"Train MAE: {epoch_loss / len(train_loader):.4f}"
        )

    # =======================
    #  Validation
    # =======================
    model.eval()
    abs_errors = []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            y = y.to(device).float()

            feats = model(x, return_features=True)
            feats_cal = fds(
                feats=feats,
                y=y,
                bin_edges=bin_edges,
                epoch=epoch,
                train=False
            )

            preds = model.regressor(feats_cal).squeeze(-1)
            abs_errors.append(torch.abs(preds - y))

    val_mae = torch.cat(abs_errors).mean().item()

    return model, fds, bin_edges, val_mae

In [8]:
trained_models = {}
fds_modules = {}
bin_edges_map = {}
fds_results = {}

print("\nTraining on V1")
trained_models['V1'], fds_modules['V1'], bin_edges_map['V1'], fds_results['V1'] = \
    train_eval_with_fds(train_v1, val_v1, train_targets_v1, CONFIG)

print("\nTraining on V2")
trained_models['V2'], fds_modules['V2'], bin_edges_map['V2'], fds_results['V2'] = \
    train_eval_with_fds(train_v2, val_v2, train_targets_v2, CONFIG)

print("\nTraining on V3")
trained_models['V3'], fds_modules['V3'], bin_edges_map['V3'], fds_results['V3'] = \
    train_eval_with_fds(train_v3, val_v3, train_targets_v3, CONFIG)

print("\nTraining on V4")
trained_models['V4'], fds_modules['V4'], bin_edges_map['V4'], fds_results['V4'] = \
    train_eval_with_fds(train_v4, val_v4, train_targets_v4, CONFIG)



Training on V1
📊 FDS initialized:
   Feature dim: 48
   Bins: 100
   Kernel: gaussian, ks=5, sigma=2.0
   Momentum: 0.9
   Start update: epoch 1
   Start smooth: epoch 2
Epoch 00 | Train MAE: 0.3086
Epoch 01 | Train MAE: 0.2052
Epoch 02 | Train MAE: 0.5634
Epoch 03 | Train MAE: 0.2628
Epoch 04 | Train MAE: 0.2303
Epoch 05 | Train MAE: 0.2166
Epoch 06 | Train MAE: 0.2141
Epoch 07 | Train MAE: 0.2077
Epoch 08 | Train MAE: 0.1974
Epoch 09 | Train MAE: 0.1950
Epoch 10 | Train MAE: 0.1942
Epoch 11 | Train MAE: 0.1897
Epoch 12 | Train MAE: 0.1894
Epoch 13 | Train MAE: 0.1890
Epoch 14 | Train MAE: 0.1883
Epoch 15 | Train MAE: 0.1880
Epoch 16 | Train MAE: 0.1877
Epoch 17 | Train MAE: 0.1876
Epoch 18 | Train MAE: 0.1872
Epoch 19 | Train MAE: 0.1868
Epoch 20 | Train MAE: 0.1866
Epoch 21 | Train MAE: 0.1864
Epoch 22 | Train MAE: 0.1860
Epoch 23 | Train MAE: 0.1856
Epoch 24 | Train MAE: 0.1854
Epoch 25 | Train MAE: 0.1849
Epoch 26 | Train MAE: 0.1800
Epoch 27 | Train MAE: 0.1797
Epoch 28 | Train 

In [9]:
print("\nFinal Comparison (Validation MAE)")
for k, v in fds_results.items():
    print(f"{k}: MAE = {v:.4f}")




Final Comparison (Validation MAE)
V1: MAE = 0.1795
V2: MAE = 0.1754
V3: MAE = 0.2961
V4: MAE = 0.3020


In [10]:
print("\nLoading test data...")
test_dataloader = load_test_data(
    test_path=PROJECT_ROOT / "datasets_challenge" / "test",
    batchsize=512,
    pipeline=pipeline_v1,   # SAME pipeline as training
    load_anonymized_data=True
)


Loading test data...
c:\mait_g2_drift_challenge\datasets_challenge\test\test_part_001.csv
c:\mait_g2_drift_challenge\datasets_challenge\test\test_part_002.csv
c:\mait_g2_drift_challenge\datasets_challenge\test\test_part_003.csv
c:\mait_g2_drift_challenge\datasets_challenge\test\test_part_004.csv
c:\mait_g2_drift_challenge\datasets_challenge\test\test_part_005.csv
c:\mait_g2_drift_challenge\datasets_challenge\test\test_part_006.csv
c:\mait_g2_drift_challenge\datasets_challenge\test\test_part_007.csv
c:\mait_g2_drift_challenge\datasets_challenge\test\test_part_008.csv
c:\mait_g2_drift_challenge\datasets_challenge\test\test_part_009.csv
c:\mait_g2_drift_challenge\datasets_challenge\test\test_part_010.csv
c:\mait_g2_drift_challenge\datasets_challenge\test\test_part_011.csv


In [11]:
def predict_on_test_with_fds(model, fds, bin_edges, test_dataloader, device):
    model.eval()
    preds_all = []

    with torch.no_grad():
        for x, _ in test_dataloader:
            x = x.to(device)

            # Get features
            feats = model(x, return_features=True)

            # Initial prediction (no FDS)
            preds_initial = model.regressor(feats).squeeze(-1)

            # Use predicted y to apply FDS
            feats_cal = fds(
                feats=feats,
                y=preds_initial,
                bin_edges=bin_edges,
                epoch=10**9,    # force FDS active
                train=False
            )

            # Final prediction
            preds_final = model.regressor(feats_cal).squeeze(-1)

            preds_all.append(preds_final.cpu().numpy())

    return np.concatenate(preds_all)


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

for split_name in ['V1', 'V2', 'V3', 'V4']:
    print(f"\n FDS test prediction for {split_name}")

    model = trained_models[split_name]
    fds = fds_modules[split_name]
    bin_edges = bin_edges_map[split_name]

    test_preds = predict_on_test_with_fds(
        model, fds, bin_edges, test_dataloader, device
    )

    df = pd.DataFrame({
        "id": np.arange(len(test_preds)),
        "predicted_sideslip": test_preds
    })

    df.to_csv(f"predictions_FDS_{split_name}.csv", index=False)
    print(f"✓ Saved predictions_FDS_{split_name}.csv")




 FDS test prediction for V1
✓ Saved predictions_FDS_V1.csv

 FDS test prediction for V2
✓ Saved predictions_FDS_V2.csv

 FDS test prediction for V3
✓ Saved predictions_FDS_V3.csv

 FDS test prediction for V4
✓ Saved predictions_FDS_V4.csv
